In [2]:
from cloudvolume import CloudVolume
import fastremap
from tqdm.auto import tqdm
import pathlib
import json
from joblib import Parallel, delayed

In [2]:
SVOX_CLOUDPATH = "precomputed://s3://cvdb-bossdb-boss/smith2024/zebrafish/supervoxels"
CKPT_CLOUDPATH_STEM = "precomputed://s3://cvdb-bossdb-boss/smith2024/zebrafish/agglomeration_checkpoint_"
COMPONENT_PATH = pathlib.Path("./component-graphs/")

RANDOMIZE_UPLOAD_ORDER = True

In [3]:
src = CloudVolume(SVOX_CLOUDPATH, progress=False)

In [4]:
zslices_at_a_time = src.chunk_size[-1]
assert zslices_at_a_time > 1

In [7]:
def int_keys_values(dct):
    return {int(k): int(v) for k, v in dct.items()}


for component_remap_file in COMPONENT_PATH.glob("*.remap.json"):
    print(component_remap_file)
    remap_dict = json.loads(
        component_remap_file.read_text(), object_hook=int_keys_values
    )
    editcount = int(str(component_remap_file).split(".")[1])

    cloudpath = CKPT_CLOUDPATH_STEM + str(editcount)
    dst = CloudVolume(cloudpath, info=src.info, progress=False)
    dst.commit_info()

    def process_slice(z, zslices_at_a_time, src, dst, remap_dict):
        z2 = min(z + zslices_at_a_time, src.bounds.maxpt[2])
        src_data = src[:, :, z:z2]
        remapped_data = fastremap.remap(
            src_data, remap_dict, preserve_missing_labels=True
        )
        dst[:, :, z:z2] = remapped_data

    Parallel(n_jobs=2)(
        delayed(process_slice)(z, zslices_at_a_time, src, dst, remap_dict)
        for z in tqdm(range(0, src.bounds.maxpt[2], zslices_at_a_time))
    )

ModuleNotFoundError: No module named 'mipreduce'

NameError: name 'editcount' is not defined